In [1]:
%load_ext autoreload
%autoreload 2

import json
import os
import yaml
from pathlib import Path
from dask.distributed import Client
import dask.dataframe as dd
import networkx as nx
import sys
import pandas as pd


import ipycytoscape
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import intervals
import pygtrie
import seaborn as sns


# App Name
app_name = "montage-pegasus-2mass-2deg-4node" #cosmoflow cm1
app_name = "montage_pegasus-dss-1deg_node-16"
cp_dir = "/p/lustre3/pandey2/logs/Results_Checkpoint/dataflow/"+app_name+"/"


/usr/workspace/pandey2/DFT/envDFT/lib/python3.9/site-packages/dask/dataframe/__init__.py:42: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


In [2]:
use_local=False
if not use_local:
    with open(f'/g/g91/pandey2/.dftracer/configuration.yaml', 'r') as file:
        dlp_yaml = yaml.safe_load(file)
        app_root = dlp_yaml["app"]
else:
    app_root = str(Path(os.getcwd()).parent.parent)
sys.path.insert(0, app_root)

import dfanalyzer
print(dfanalyzer.__file__)
from dfanalyzer.main import DFAnalyzer,get_dft_configuration,update_dft_configuration,setup_logging,setup_dask_cluster, reset_dask_cluster, get_dft_configuration
from dfanalyzer.graph_visualization.cytoscape import GraphFunctions, CytoGraph
from dfanalyzer.graph1 import DFGrepInterferencePartitionBased, DFGrepBurstiness

/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/__init__.py


In [3]:
use_local=False
if not use_local:
    with open(f'/g/g91/pandey2/.dftracer/configuration.yaml', 'r') as file:
        dlp_yaml = yaml.safe_load(file)
        app_root = dlp_yaml["app"]
else:
    app_root = str(Path(os.getcwd()).parent.parent)
sys.path.insert(0, app_root)

import dfanalyzer
print(dfanalyzer.__file__)
from dfanalyzer.main import DFAnalyzer,get_dft_configuration,update_dft_configuration,setup_logging,setup_dask_cluster, reset_dask_cluster, get_dft_configuration
from dfanalyzer.graph_visualization.cytoscape import GraphFunctions, CytoGraph
from dfanalyzer.graph1 import DFGrepInterferencePartitionBased, DFGrepBurstiness, DFGrepWorkflow, DFGrepWorkflow1

if not use_local:
    dask_run_dir = os.path.join(app_root, "dfanalyzer", "dask", "run_dir")
    with open (os.path.join(dask_run_dir, f"scheduler_{os.getenv('USER')}.json"), "r") as f:
        dask_scheduler = json.load(f)["address"]
else:
    dask_scheduler = None

# App Name
app_name = "montage_pegasus-dss-1deg_node-4"

cp_dir = "/p/lustre3/pandey2/logs/Results_Checkpoint/dataflow/"+app_name+"/"
os.makedirs(cp_dir, exist_ok=True)

condition_fn = None #

if app_name == "montage_pegasus-dss-1deg_node-4":
    # filename = "/p/lustre3/pandey2/logs/RAW_copy/montage_pegasus-dss-1deg_node-16/*.pfw.gz"
    # # filename = "/p/lustre3/iopp/dftracer-traces-lfs/v1.0.5-develop/corona/montage/pegasus-dss-2deg/node-16/v1/COMPACT/*.pfw.gz"
    # filename = "/p/lustre3/pandey2/logs/RAW_copy/montage_pegasus-dss-1deg_node-16/COMPACT/*.pfw.gz"
    # filename = "/p/lustre3/pandey2/logs/RAW_copy/montage_pegasus-dss-1deg_node-16/RAW/*.pfw.gz"
    # filename = "/p/lustre3/pandey2/logs/RAW_copy/montage_pegasus-dss-1deg_node-4/RAW/*.pfw.gz"
    filename = "/p/lustre3/pandey2/logs/RAW_copy/montage_pegasus-dss-1deg_node-4/COMPACT/*.pfw.gz"


else:
    raise Exception("Unknown App name")


# Configuration 4 update log file dlp -> df
conf = update_dft_configuration(dask_scheduler=dask_scheduler, verbose=True, debug=True,
                                log_file=f"./dft_{os.getenv('USER')}.log", rebuild_index=False, time_approximate=True, 
                                host_pattern=r'lassen(\d+)', time_granularity=1e6, skip_hostname=True, conditions=condition_fn)
conf = get_dft_configuration()


# Setup
setup_logging()
setup_dask_cluster()
reset_dask_cluster()



[INFO] [10:11:41] Initialized Client with 288 workers and link http://134.9.71.27:37945/status [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:770]


/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/__init__.py


[INFO] [10:11:59] Restarting all workers [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:762]


In [5]:
def all_mount_points():
    with open("/proc/mounts", "r") as file:
        mount_points = [line.split()[1] for line in file]
    with open("/usr/workspace/pandey2/lassen_mounts", "r") as file:
        mount_p = [line.split()[1] for line in file]
    return mount_points+mount_p

mount_points = all_mount_points()
trie = pygtrie.StringTrie(zip(mount_points, [True] * len(mount_points)))

def cols_function(json_object, current_dict, time_approximate,condition_fn,load_data):
    d = {}
    def find_mount_point(path,trie):
        mount_point = trie.longest_prefix(path)
        if mount_point:
            return mount_point.key
        return '/'.join(path.split('/', 3)[:3])

    if "M" == json_object["ph"] and "FH" == json_object["name"] and "args" in json_object and "name" in json_object["args"]:
        d["mount_point"] = find_mount_point(trie=load_data["mount_point"],path=json_object["args"]["name"])
    if "args" in json_object and "M" != json_object["ph"]:
        if "ret" in json_object["args"]:
            d["size"] = int(json_object["args"]["ret"]) 

    if "name" in json_object:
        if (json_object["name"] in ["fwrite", "write","pwrite","fputs"]):
            d["prod"] = 1
            d["cons"] = 0
        else:
            d["prod"] = 0
            d["cons"] = 1
    return d

load_cols = {'size': "int64[pyarrow]", 'prod':"uint16[pyarrow]", 'cons':"uint16[pyarrow]" }
load_cols_metadata = {"FH":{'mount_point':"string[pyarrow]" }}


In [6]:
analyzer = DFAnalyzer(filename,load_fn=cols_function, load_cols=load_cols, load_data={"mount_point":trie}, metadata_cols = load_cols_metadata)
# analyzer = DFAnalyzer(filename)

[INFO] [10:12:25] Created index for 1 files [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:429]
[INFO] [10:12:25] Total size of all files are <dask.bag.core.Item object at 0x15554173beb0> bytes [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:431]
[INFO] [10:12:25] test debug [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:434]
[INFO] [10:12:27] Loading 993 batches out of 1 files and has 16262272 lines overall [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:445]
[INFO] [10:13:08] Loaded events [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:512]
[INFO] [10:13:08] Loaded plots with slope threshold: 45 [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:518]


In [8]:
# if app_name == "montage_pegasus-dss-1deg_node-16":
if True:
    df1 = analyzer.events[~analyzer.events['fhash'].isna()] 
    df2 = analyzer.file_hash.reset_index()[["hash", "mount_point"]]
    df3 = analyzer.host_hash.reset_index()[['hhash', 'name']] 
    # # df3 = df3.rename(columns={'hash':'hhash'})
    result_df = df1.merge(df2, left_on="fhash", right_on="hash", how='left').drop(columns = ['hash_x', 'hash_y', 'mount_point_x']).rename(columns={'mount_point_y': 'mount_point'})
    result_df1 = result_df.merge(df3, on="hhash", how='left', suffixes=('_left', '_right')).rename(columns={'name_left': 'name', 'name_right':'hostname'})
    analyze_df = result_df1[['name','cat','size','ts','te','dur','trange','mount_point','hostname','fhash','pid','tid','prod','cons']] 
    analyze_df["mount_point"] = analyze_df["mount_point"].where(analyze_df["mount_point"].str.startswith("/"), "/p/lustre3")


In [32]:
analyze_df.compute()

,name,cat,size,ts,te,dur,trange,mount_point,hostname,fhash,pid,tid,prod,cons
0,fopen,STDIO,<NA>,499468,499505,37,0,/proc,corona174,1.685811701929079e+19,350380,350380,0,1
1,fclose,STDIO,<NA>,500954,500965,11,0,/proc,corona174,1.685811701929079e+19,350380,350380,0,1
2,opendir,POSIX,<NA>,503191,503221,30,0,/sys,corona174,8.594106316560531e+18,350380,350380,0,1
3,fopen,STDIO,<NA>,505598,505633,35,0,/sys,corona174,8.88885951859612e+18,350380,350380,0,1
4,fclose,STDIO,<NA>,507034,507043,9,0,/sys,corona174,8.88885951859612e+18,350380,350380,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
509424,fwrite,STDIO,17280,451033529,451033640,111,451,/p/lustre3,corona171,6.010016916925533e+18,590746,590746,1,0
509425,fwrite,STDIO,2880,451034495,451034496,1,451,/p/lustre3,corona171,5.530042356080299e+18,590746,590746,1,0
509426,fclose,STDIO,<NA>,451035381,451036550,1169,451,/p/lustre3,corona171,5.530042356080299e+18,590746,590746,0,1
509427,fwrite,STDIO,2880,451037497,451037498,1,451,/p/lustre3,corona171,6.010016916925533e+18,590746,590746,1,0


In [9]:
wf = DFGrepWorkflow(analyze_df)
wf.select_events()

# lvl1_results = wf.get_wfGraph().compute()
# lvl2_results = wf.get_wfGraph(level=2).compute()
# lvl3_results = wf.get_wfGraph(level=3).compute()

lvl1_results = wf.get_wfGraph()
lvl2_results = wf.get_wfGraph(level=2)
lvl3_results = wf.get_wfGraph(level=3)

In [11]:
lvl3_results.drop_duplicates().query("prod > 0").compute()

,fhash,hostname,prod,cons


In [53]:
lvl3_results.groupby("prod").count().compute()

,fhash,hostname,cons
prod,,,
0,15134614,15134614,15134614
1,4048,4048,4048


In [56]:
lvl3_results.drop_duplicates().groupby("prod").count().compute()

,fhash,hostname,cons
prod,,,
0,12950,12950,12950
1,1,1,1


# Graphs

In [107]:
from functions import *
df3_unique_edges = lvl3_results.drop_duplicates()
df2_unique_edges = lvl2_results.drop_duplicates()
df1_unique_edges = lvl1_results.drop_duplicates()
# (A) Original: per (fhash, hostname)
l3_results = compute_io_metrics_host(df3_unique_edges, analyze_df, compute_result=False)

# (B) New: per (fhash, pid) using df2 (which has columns ['fhash','pid', ...])
l2_results = compute_io_metrics_pid(df2_unique_edges, analyze_df, compute_result=False)

l1_results = compute_io_metrics_pid(df1_unique_edges, analyze_df, compute_result=False)


In [108]:
l3_results.compute()

,fhash,hostname,prod,cons,io_time,io_count,io_size,iops,bw
0,1.685811701929079e+19,corona174,0,1,307,192,0,0.625407,0.0
1,8.594106316560531e+18,corona174,0,1,152,48,0,0.315789,0.0
2,8.88885951859612e+18,corona174,0,1,201,96,0,0.477612,0.0
3,1.5508008208381696e+19,corona174,0,1,184,96,0,0.521739,0.0
4,9.153441086994398e+18,corona174,0,1,511,48,0,0.093933,0.0
...,...,...,...,...,...,...,...,...,...
12946,1.6961543819921609e+19,corona173,0,1,10852300,2590,53174436,0.000239,4.899831
12947,7.614793105686147e+18,corona175,0,1,143,3,0,0.020979,0.0
12948,1.4747597561773287e+19,corona175,0,1,11827,2,12,0.000169,0.001015
12949,3.963517827694177e+18,corona175,0,1,46271,2,13,0.000043,0.000281


In [109]:
l3_results.query("io_size > 0 and prod > 0").compute()

,fhash,hostname,prod,cons,io_time,io_count,io_size,iops,bw
12164,10533921333604984978,corona173,1,0,302189,4048,38854080,0.013396,128.575428


In [85]:
lvl3_results.query("prod > 0").fhash.unique().compute()

0    10533921333604984978
Name: fhash, dtype: string

In [110]:
f1 = analyzer.file_hash.reset_index()[['hash','name']]

a1 = l3_results.merge(f1, left_on="fhash", right_on="hash", how='left')
a1.query("prod > 0").compute()

,fhash,hostname,prod,cons,io_time,io_count,io_size,iops,bw,hash,name
12164,10533921333604984978,corona173,1,0,302189,4048,38854080,0.013396,128.575428,<NA>,<NA>


In [93]:
a1.head()

,fhash,hostname,prod,cons,io_time,io_count,io_size,iops,bw,hash,name
0,1.685811701929079e+19,corona174,0,1,307,192,0,0.625407,0.0,1.685811701929079e+19,/proc/self/status
1,8.594106316560531e+18,corona174,0,1,152,48,0,0.315789,0.0,8.594106316560531e+18,/sys/devices/system/node
2,8.88885951859612e+18,corona174,0,1,201,96,0,0.477612,0.0,8.88885951859612e+18,/sys/devices/system/node/node0/meminfo
3,1.5508008208381696e+19,corona174,0,1,184,96,0,0.521739,0.0,1.5508008208381696e+19,/sys/devices/system/node/node1/meminfo
4,9.153441086994398e+18,corona174,0,1,511,48,0,0.093933,0.0,9.153441086994398e+18,/g/g92/kogiou1/.mvapich2.conf


In [95]:
a1.fhash.nunique().compute()

np.int64(9255)

In [111]:
a1.loc[12160:12167].compute()

,fhash,hostname,prod,cons,io_time,io_count,io_size,iops,bw,hash,name
12160,5249841752869457710,corona173,0,1,8264282,1350,27853200,0.000163,3.370311,<NA>,<NA>
12161,15522895521084426727,corona173,0,1,8217628,1350,27853200,0.000164,3.389445,<NA>,<NA>
12162,2168939933086450,corona173,0,1,8166744,1350,27842400,0.000165,3.409241,<NA>,<NA>
12163,8654951505122374458,corona173,0,1,8229139,1350,27842400,0.000164,3.383392,<NA>,<NA>
12164,10533921333604984978,corona173,1,0,302189,4048,38854080,0.013396,128.575428,<NA>,<NA>
12165,17827443556906684962,corona173,0,1,882322,721,14849280,0.000817,16.829774,<NA>,<NA>
12166,13505406581113878810,corona173,0,1,876106,721,14849280,0.000823,16.949182,<NA>,<NA>
12167,5.783190811416891e+18,corona171,0,1,56014,6,13,0.000107,0.000232,5.783190811416891e+18,merge_whole-wf.in.err.22


#  Check

In [113]:
analyzer.events.fhash.nunique().compute()
analyzer.file_hash.reset_index().hash.nunique().compute()

np.int64(9414)

In [117]:
analyzer.file_hash.reset_index().hash.nunique().compute()

np.int64(12282)

In [7]:
# 1️⃣ Extract the unique values
fhash_values = analyzer.events["fhash"].dropna().unique()
hash_values = analyzer.file_hash.reset_index()["hash"].dropna().unique()

# 2️⃣ Compute them
fhash_set = set(fhash_values.compute())
hash_set = set(hash_values.compute())

# 3️⃣ Find which fhash values are missing in file_hash
missing_fhash = fhash_set - hash_set

# 4️⃣ Print results
print(f"Total fhash in events: {len(fhash_set)}")
print(f"Total hash in file_hash: {len(hash_set)}")
print(f"Missing fhash count: {len(missing_fhash)}")

if missing_fhash:
    print("Example missing fhash values:", list(missing_fhash)[:10])
else:
    print("✅ All fhash values from events are present in file_hash.")

Total fhash in events: 9398
Total hash in file_hash: 12282
Missing fhash count: 0
✅ All fhash values from events are present in file_hash.
